# Modes of photonic bandgap fibers

```{index} photonic bandgap fiber
```
```{index} PBG
```

*Photonic bandgap* (PBG) fibers guide light by a mechanism distinct from both total internal reflection of step-index fibers and the antiresonance of ARF. A periodic arrangement of high-index strands in the cladding creates photonic bandgaps, which are ranges of effective index in which transverse propagation through the cladding lattice is forbidden. A defect core (here formed by *omitting* a few strands) then supports modes whose effective indices fall inside a bandgap. Such modes are prevented from fully radiating through the cladding even though the core index is *not* higher than its surroundings [[Litchinitser et al 2002](#references)].

The `PBG` class builds all-solid bandgap fibers from a parameter dictionary specifying the strand lattice (pitch $\Lambda$, strand radius, number of layers, indices) together with jacket and PML data. Ready-made dictionaries for several published fibers are provided in `fibermode.pbg.fiber_dicts`.

```{index} photonic crystal fiber
```
```{index} PCF
```
```{index} PBG; vs PCF
```

Note that *photonic bandgap fiber* is not synonymous with the broader term *photonic crystal fiber* (PCF): PCF covers any fiber with a periodic lattice microstructure in the cladding, including ordinary *index-guided* ("holey") fibers that merely lower the cladding's average index and guide by a conventional total-internal-reflection-like mechanism -- no bandgap involved. Only fibers like those in this chapter guide via an actual bandgap effect. The `PBG` class's geometry machinery can build either kind; see `demos/pbg/holey_demo.py` for a worked index-guided (non-bandgap) example built with the same class.

In [ ]:
import ngsolve as ng
import numpy as np
from ngsolve.webgui import Draw
from fibermode import PBG
from fibermode.pbg.fiber_dicts.lyr6cr2 import params

## Constructing `PBG` objects

The `lyr6cr2` dictionary describes a fiber with six rings of high-index strands and a two-ring core defect:

In [ ]:
params

In [ ]:
A = PBG(params)
Draw(A.mesh);

The refractive index profile is exposed as the coefficient function `N` (and the derived potential-like coefficient `V` used in the eigenproblem):

In [ ]:
Draw(A.N, A.mesh, 'index', settings={"Objects": {"Wireframe": False, "Edges":False}})
A.n_tube, A.n_clad

Two conveniences worth knowing are `A.rotate(angle)`, which rotates the strand lattice (useful for studying orientation sensitivity), and `A.reset_mesh()` which restores the original. Here `A.N` may be modified (e.g., thermally perturbed) and restored by `A.reset_N()`:

```{index} thermal perturbation
```
```{index} perturbation; index
```
```{index} rotate; PBG fiber
```

In [ ]:
# example: a smooth thermal perturbation of the index profile
T = ng.exp(-(ng.x**2 / 30 + ng.y**2 / 30))
A.N = A.N + 0.6 * T
Draw(A.N, A.mesh, 'perturbed index', 
     settings={"Objects": {"Wireframe": False, "Edges":True}})
A.reset_N()

## Computing leaky modes

Modes of the defect core are leaky: they can tunnel through the finite number of cladding rings. We search the $Z$-plane using the polynomial (frequency-dependent) PML formulation, with a thin elliptical contour around the fundamental mode group, following `demos/pbg/PBG_mode.py`:

```{index} leakymode; PBG
```

**Tip: finding a search center systematically.** Rather than guessing `ctr`/`rad` by trial and error, `A.sqrZfrom(beta)` converts a target physical propagation constant into the corresponding nondimensional $Z^2$, the coordinate FEAST searches in. A principled first guess for a core-guided mode is a $\beta$ just below the cladding's own index-guided cutoff, e.g. `A.sqrZfrom(.9998 * A.k * A.n_core)`; a wide-radius search centered there (as below) then locates the actual bandgap-guided modes nearby. (Note this assumes the outer/PML region shares the cladding index, `n0 == n_clad`. This is not the case for every fiber_dict, e.g. `lyr6cr2_w_air` uses an air outer region instead.)


```{index} search center; principled guess 
```
```{index} eigenvalue search; FEAST
```

In [ ]:
z, y, yl, beta, P, extras = A.leakymode(
    3,                     # finite element degree
    ctr=2, rad=1,          # elliptic contour in the Z-plane
    rhoinv=0.8,
    quadrule='ellipse_trapez_shift',
    alpha=A.alpha,
    npts=6, nspan=3,
    niterations=12, nrestarts=0,
    stop_tol=1e-8,
    seed=1,
)

In [ ]:
print('Z    =', z)
print('beta =', beta)
print('CL [dB/m] =', 20 * beta.imag / np.log(10))

In [ ]:
Draw(ng.Norm(y.gridfun())**2, A.mesh, settings={"Objects": {"Wireframe": False}});

The mode intensity is trapped in the low-index core defect, with the strand lattice acting as the bandgap mirror — visibly different from the index-guided modes of earlier notebooks.

## Custom strand patterns

The lattice need not be complete. The `pattern` entry of the parameter dictionary lists, ring by ring, which strand sites are occupied (1) or vacant (0). Here is an example keeping the two innermost rings complete and opening channels in the third ring:

```{index} pattern; custom PBG
```
```{index} PBG; custom pattern
```

In [ ]:
pattern = [
    [1] * 12,          # ring 1: complete
    [1] * 18,          # ring 2: complete
    [1, 1, 0, 1] * 6,  # ring 3: one vacancy per sector
]
params2 = dict(params)
params2['pattern'] = pattern
params2['layers'] = len(pattern)
B = PBG(params2)
Draw(B.mesh);

Such patterned claddings model coupling channels, polarization- breaking designs, or fabrication defects; the mode-solving calls are unchanged.

Other ready-made parameter dictionaries in `fibermode.pbg.fiber_dicts` include `lyr7cr1` (seven rings, one-ring core), `holey` (air-hole fiber), and `rod`.

<a id='references'></a>
## References

- N. M. Litchinitser, A. K. Abeeluck, C. Headley, and B. J. Eggleton. *Antiresonant reflecting photonic crystal optical waveguides.* Optics Letters, 27:1592-1594, 2002.
- F. Luan et al. *All-solid photonic bandgap fiber.* Optics Letters, 29:2369-2371, 2004.
- A. Argyros, T. Birks, S. Leon-Saval, C. M. B. Cordeiro, and P. St. J. Russell. *Guidance properties of low-contrast photonic bandgap fibres.* Optics Express, 13:2503-2511, 2005.